# Delta Lake smoke test on MinIO

Open in VS Code with your repo's Python kernel. Install `pyspark==3.5.6` and `python-dotenv` in that kernel if needed, then restart it. Use JDK 17 for Spark 3.5.6.

Create `data-platform/.env` with `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`. Keep it ignored by Git. Open the repository in VS Code and run all cells. The `bronze` bucket must exist. Each run uses a new path under `bronze/_smoke/`.

In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
from pathlib import Path
from uuid import uuid4

from dotenv import dotenv_values
from pyspark.sql import SparkSession
from py4j.protocol import Py4JJavaError


cwd = Path.cwd().resolve()
env_file = next(
    (folder / ".env" for folder in (cwd, *cwd.parents)
     if folder.name == "data-platform"),
    cwd / "data-platform" / ".env",
)
if not env_file.is_file():
    raise FileNotFoundError(f"Create {env_file} and open the repo in VS Code.")

credentials = dotenv_values(env_file, interpolate=False)
required = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")
missing = [name for name in required if not credentials.get(name)]
if missing:
    raise ValueError(f"Missing credentials in {env_file}: {', '.join(missing)}")

for name in required:
    os.environ[name] = credentials[name]
del credentials

os.environ["PYSPARK_PYTHON"] = sys.executable

try:
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("reconciliation-delta-smoke-test")
        .config("spark.jars.packages",
                "io.delta:delta-spark_2.12:3.3.3,org.apache.hadoop:hadoop-aws:3.3.4")
        .config("spark.jars.ivy", str(Path.home() / ".ivy2"))
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.shuffle.partitions", "2")
        .config("spark.databricks.delta.snapshotPartitions", "2")
        .config("spark.hadoop.fs.s3a.endpoint", "http://192.168.29.250:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
        .getOrCreate()
    )
except Py4JJavaError as exc:
    cause = exc.java_exception
    while cause is not None:
        print(cause.toString())
        cause = cause.getCause()

26/11/17 05:15:04 WARN Utils: Your hostname, DESKTOP-8SSCKF6 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/17 05:15:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/kirit/.venvs/bronze-py311/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/kirit/.ivy2/cache
The jars for the packages stored in: /home/kirit/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1a3d23c2-d493-46ab-9f73-9513ddaa9420;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 458ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.3.3 from central in [default]
	io.delta#delta-storage;3.3.3 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop

In [3]:
table_path = f"s3a://bronze/_smoke/delta-{uuid4().hex}"
expected = [(1, "write"), (2, "read")]

# Generate the test rows in Spark SQL.
source = spark.sql("SELECT * FROM VALUES (1, 'write'), (2, 'read') AS data(id, label)")
source.write.format("delta").mode("errorifexists").save(table_path)

result = spark.read.format("delta").load(table_path).orderBy("id")
actual = [(row.id, row.label) for row in result.collect()]
if actual != expected:
    raise RuntimeError(f"Expected {expected}, got {actual}")

result.show()
print(f"PASS: wrote and read {len(actual)} Delta rows at {table_path}")

26/11/17 05:15:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/11/17 05:16:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+-----+
| id|label|
+---+-----+
|  1|write|
|  2| read|
+---+-----+

PASS: wrote and read 2 Delta rows at s3a://bronze/_smoke/delta-a4bd066e8bfd4096b64ae164917f5f40
